# Landau window and tail certificates

Run from the repository root, or upload the four implementation files when prompted by Colab. This notebook checks each module hash before running the same implementation as the command line. The fixed MB-06 ratio is an arithmetic comparison case. The scope is the bounded calculation and threshold decisions.

In [ ]:
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

pins = {'window_kernel.py': '9c7c14b2fb5539ff450fed8ee8c8d65117396c1389d5cc2a074891854881502f', 'riemann_evidence/__init__.py': '6ca64d9c498deddb106dad0b72d99e49a42935099fe18298879f7e9f82507cb8', 'riemann_evidence/rational_intervals.py': '5aa5fdcbb17ca443ddaa9c3362781da98b4a03d52afe0190fe1677a4a81acab9', 'riemann_evidence/verification.py': '6fd038c01f72b0ec8cce14396320e50fa8fe6c047232c60d1b40ed873ae876ac'}

root = Path.cwd()
if not all((root / path).is_file() for path in pins):
    from google.colab import files

    print("Select window_kernel.py and the three files in riemann_evidence/.")
    uploads = files.upload()
    wanted = {Path(path).name for path in pins}
    if set(uploads) != wanted:
        raise ValueError(f"Expected exactly {sorted(wanted)}")
    _runtime = TemporaryDirectory(prefix="riemann-evidence-")
    root = Path(_runtime.name)
    for path in pins:
        target = root / path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(uploads[target.name])

for path, digest in pins.items():
    if sha256((root / path).read_bytes()).hexdigest() != digest:
        raise ValueError(f"Local implementation hash mismatch: {path}")


In [ ]:
import json
import subprocess

completed = subprocess.run(
    [sys.executable, "-E", "-S", str(root / "window_kernel.py")],
    check=True,
    capture_output=True,
)
report = json.loads(completed.stdout)
data = (json.dumps(report, indent=2) + "\n").encode("utf-8")
Path("window-results-v2.json").write_bytes(data)
for name, result in report.items():
    print(name, result["status"], result["details"].get("fractional_bits", "-"))
print("Report SHA-256:", sha256(data).hexdigest())
